# Cookbook: XBRL Financial Analysis

Deep dive into SEC XBRL data using `python-sec`. This notebook demonstrates
how to retrieve, explore, and analyze structured financial data from the
XBRL company facts API.

**What you'll learn:**
1. Browse available taxonomies and concepts
2. Retrieve specific financial metrics (Revenue, Assets, EPS)
3. Filter by unit of measure and fiscal period
4. Build time-series DataFrames for trend analysis
5. Use the XBRL frames endpoint for cross-company comparisons

## 1. Setup

In [ ]:
from edgar.client import EdgarClient

edgar_client = EdgarClient(user_agent="Your Name your-email@example.com")

## 2. Load XBRL Facts for a Company

`get_facts()` returns a `Facts` model wrapping all XBRL data a company has filed.
The SEC organizes facts by taxonomy (e.g. `us-gaap`, `dei`, `ifrs-full`).

In [ ]:
company = edgar_client.company("AAPL")
facts = company.get_facts()

print(f"Entity:     {facts.entity_name}")
print(f"CIK:        {facts.cik}")
print(f"Taxonomies: {facts.taxonomies}")

## 3. Explore Available Concepts

Each taxonomy contains hundreds of concepts. Use `concepts()` to list them
and `label()` / `description()` for metadata.

In [ ]:
# How many concepts are available?
gaap_concepts = facts.concepts("us-gaap")
dei_concepts = facts.concepts("dei")

print(f"us-gaap concepts: {len(gaap_concepts)}")
print(f"dei concepts:     {len(dei_concepts)}")

In [ ]:
# Browse some common financial concepts.
key_concepts = [
    "Revenues",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "EarningsPerShareBasic",
    "StockholdersEquity",
]

print(f"{'Concept':<30} {'Label':<40} {'Units'}")
print("-" * 90)
for concept in key_concepts:
    try:
        label = facts.label("us-gaap", concept)
        units = facts.units("us-gaap", concept)
        print(f"{concept:<30} {label:<40} {units}")
    except (KeyError, IndexError):
        print(f"{concept:<30} (not reported by this company)")

## 4. Retrieve a Single Concept

`facts.get(taxonomy, concept)` returns a `list[Fact]` sorted by end date.
Each `Fact` has `value`, `end`, `fiscal_year`, `fiscal_period`, `form`, and `filed`.

In [ ]:
revenue = facts.get("us-gaap", "Revenues")

print(f"Total Revenue data points: {len(revenue)}\n")
print(f"{'End Date':<14} {'FY':>4} {'Period':<4} {'Value':>18} {'Form':<8} {'Filed'}")
print("-" * 75)
for r in revenue[-10:]:
    print(f"{r.end:<14} {r.fiscal_year:>4} {r.fiscal_period:<4} {r.value:>18,.0f} {r.form:<8} {r.filed}")

## 5. Filter by Unit of Measure

Some concepts report in multiple units. Use the `unit` parameter to filter.

In [ ]:
# EPS is reported in USD/shares.
eps_units = facts.units("us-gaap", "EarningsPerShareBasic")
print(f"Available units for EPS: {eps_units}")

eps = facts.get("us-gaap", "EarningsPerShareBasic", unit="USD/shares")
print(f"\nEPS data points: {len(eps)}")
for e in eps[-5:]:
    print(f"  FY{e.fiscal_year} {e.fiscal_period}: {e.value}")

## 6. Annual vs. Quarterly Data

XBRL facts include both annual (`10-K`) and quarterly (`10-Q`) filings.
Filter by `fiscal_period` or `form` to separate them.

In [ ]:
# Annual revenue only (from 10-K filings).
annual_revenue = [r for r in revenue if r.form == "10-K"]
quarterly_revenue = [r for r in revenue if r.form == "10-Q"]

print(f"Annual data points:    {len(annual_revenue)}")
print(f"Quarterly data points: {len(quarterly_revenue)}")

print("\nAnnual Revenue:")
for r in annual_revenue[-5:]:
    print(f"  FY{r.fiscal_year}: ${r.value:>15,.0f}")

## 7. Time-Series DataFrame

Convert facts to a pandas DataFrame for analysis, charting, or export.

In [ ]:
revenue_df = facts.to_dataframe("us-gaap", "Revenues", unit="USD")
revenue_df.tail(10)

In [ ]:
# Filter to annual periods and compute year-over-year growth.
annual_df = revenue_df[revenue_df["form"] == "10-K"].copy()
annual_df["value_numeric"] = annual_df["value"].astype(float)
annual_df["yoy_growth"] = annual_df["value_numeric"].pct_change()

annual_df[["end", "fiscal_year", "value_numeric", "yoy_growth"]].tail(10)

## 8. Multi-Metric Dashboard

Pull several concepts at once to build a financial snapshot.

In [ ]:
metrics = {
    "Revenues": "USD",
    "NetIncomeLoss": "USD",
    "Assets": "USD",
    "Liabilities": "USD",
    "EarningsPerShareBasic": "USD/shares",
}

print(f"{'Metric':<28} {'Latest Value':>18} {'Period':<10} {'Filed'}")
print("-" * 75)
for concept, unit in metrics.items():
    try:
        data = facts.get("us-gaap", concept, unit=unit)
        latest = data[-1]
        if isinstance(latest.value, (int, float)) and abs(latest.value) > 100:
            value_str = f"{latest.value:>18,.0f}"
        else:
            value_str = f"{latest.value:>18}"
        print(f"{concept:<28} {value_str} FY{latest.fiscal_year:<6} {latest.filed}")
    except (KeyError, IndexError):
        print(f"{concept:<28} {'N/A':>18}")

## 9. XBRL Frames — Cross-Company Comparison

The frames endpoint returns one fact per company for a specific concept and period.
This lets you compare a metric across all filers.

In [ ]:
xbrl_service = edgar_client.xbrl()

# Get Revenue for all companies in calendar year 2023.
frames_data = xbrl_service.frames(
    concept="Revenues",
    unit_of_measure="USD",
    period="CY2023",
    taxonomy="us-gaap",
)

if frames_data:
    print(f"Taxonomy: {frames_data.get('taxonomy', 'N/A')}")
    print(f"Concept:  {frames_data.get('tag', 'N/A')}")
    print(f"Period:   {frames_data.get('ccp', 'N/A')}")
    print(f"Companies reporting: {len(frames_data.get('data', []))}")

## 10. Direct XBRL Service Access

For advanced use cases, access the XBRL service directly with `company_concepts()`.

In [ ]:
# Get all Revenue disclosures for Apple via the low-level API.
concepts_data = xbrl_service.company_concepts(
    cik="320193",
    concept="Revenues",
    taxonomy="us-gaap",
)

if concepts_data:
    print(f"Entity: {concepts_data.get('entityName', 'N/A')}")
    units = concepts_data.get("units", {})
    for unit_name, values in units.items():
        print(f"\nUnit: {unit_name} ({len(values)} data points)")
        for v in values[-3:]:
            print(f"  {v.get('end', 'N/A')}: {v.get('val', 'N/A'):>15,}")

## Summary

| Task | Method | Returns |
|------|--------|---------|
| Load all facts | `company.get_facts()` | `Facts` |
| List taxonomies | `facts.taxonomies` | `list[str]` |
| List concepts | `facts.concepts("us-gaap")` | `list[str]` |
| Get data points | `facts.get("us-gaap", "Revenues")` | `list[Fact]` |
| Filter by unit | `facts.get(..., unit="USD")` | `list[Fact]` |
| Concept metadata | `facts.label(...)`, `facts.description(...)` | `str` |
| Export to DataFrame | `facts.to_dataframe("us-gaap", "Revenues")` | `DataFrame` |
| Cross-company data | `xbrl.frames("Revenues", "USD", "CY2023")` | `dict` |